# SVD Baseline — H&M Sampled Dataset

Adapts the SVD++ template to the H&M sampled data (cached from `hm_sampled/`).
Columns: `customer_id`, `product_code`, `bought` (purchase count).
Uses the same train / val / test split generated by the sampling pipeline.

In [2]:
import pandas as pd
import torch
import numpy as np
import math
import os
from sklearn.utils import shuffle

## 1. Load Sampled H&M Data

In [4]:
# ── Load sampled H&M dataset from cache ───────────────────────────────────────
TARGET_USERS     = 1000
MIN_INTERACTIONS = 20
N_QUANTILES      = 10
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_INTERACTIONS}_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cache not found in '{SAMPLED_DATA_DIR}/'. "
    "Run the sampling notebook first."
)

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded: {cache_tag}")
print(f"Users: {n_users} | Items: {n_items}")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
train_df.head()

Loaded: u1000_m20_s42
Users: 948 | Items: 7418
Train: 38708 | Val: 9677 | Test: 15696


,customer_id,product_code,bought
0,115,339,4
1,634,546,1
2,630,230,2
3,233,222,2
4,778,9,1


## 2. Build Rating Matrices

The SVD model operates on a full `n_users × n_items` dense matrix.
Missing entries are filled with `-1` (masked out in the loss).

In [6]:
# ── Scale ratings to [0, 1] based on train set min/max ───────────────────────
min_rating = float(train_df["bought"].min())
max_rating = float(train_df["bought"].max())

def make_matrix(df, n_users, n_items, min_r, max_r):
    """
    Dense n_users x n_items tensor.
    Observed entries scaled to [0, 1]; missing entries = -1.
    Guard: if max_r == min_r (all values identical) set all observed to 0.5
    to avoid division by zero producing NaN.
    """
    mat   = torch.full((n_users, n_items), -1.0)
    users = torch.tensor(df["customer_id"].values, dtype=torch.long)
    items = torch.tensor(df["product_code"].values, dtype=torch.long)
    if max_r > min_r:
        vals = torch.tensor(
            (df["bought"].values - min_r) / (max_r - min_r), dtype=torch.float32
        )
    else:
        # All values identical — map to 0.5 (mid-point of [0,1])
        vals = torch.full((len(df),), 0.5, dtype=torch.float32)
    mat[users, items] = vals
    return mat

train_matrix = make_matrix(train_df, n_users, n_items, min_rating, max_rating)
val_matrix   = make_matrix(val_df,   n_users, n_items, min_rating, max_rating)
test_matrix  = make_matrix(test_df,  n_users, n_items, min_rating, max_rating)

# Sanity check: no NaN values should exist
for name, mat in [("train", train_matrix), ("val", val_matrix), ("test", test_matrix)]:
    n_nan = torch.isnan(mat).sum().item()
    assert n_nan == 0, f"{name} matrix contains {n_nan} NaN values!"

print(f"min_rating={min_rating}, max_rating={max_rating}")
print(f"Train : {n_users} x {n_items}, {(train_matrix != -1).sum().item()} observed")
print(f"Val   : {(val_matrix   != -1).sum().item()} observed")
print(f"Test  : {(test_matrix  != -1).sum().item()} observed")


min_rating=1.0, max_rating=20.0
Train : 948 x 7418, 38708 observed
Val   : 9677 observed
Test  : 15696 observed


## 3. SVD Model & Loss

In [8]:
class SVD(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, u_features, v_features, u_bias, v_bias):
        # u_features: (n_users, K)  v_features: (n_items, K)
        # u_bias:     (n_users,)    v_bias:     (n_items,)
        predicted = torch.sigmoid(
            torch.mm(u_features, v_features.t())          # (n_users, n_items)
            + u_bias.unsqueeze(1).expand(-1, n_items)     # row-wise user bias
            + v_bias.unsqueeze(0).expand(n_users, -1)     # col-wise item bias
        )
        return predicted


class SVDLoss(torch.nn.Module):
    def __init__(self, lam_u=0.1, lam_v=0.1, lam_1=0.1, lam_2=0.1):
        super().__init__()
        self.lam_u = lam_u
        self.lam_v = lam_v
        self.lam_1 = lam_1
        self.lam_2 = lam_2

    def forward(self, matrix, predicted, u_features, v_features, u_bias, v_bias):
        mask = (matrix != -1).float()
        prediction_error   = torch.sum(((matrix - predicted) ** 2) * mask)
        u_reg  = self.lam_u * torch.sum(u_features.norm(dim=1))
        v_reg  = self.lam_v * torch.sum(v_features.norm(dim=1))
        ub_reg = self.lam_1 * u_bias.norm()
        vb_reg = self.lam_2 * v_bias.norm()
        return prediction_error + u_reg + v_reg + ub_reg + vb_reg


def compute_rmse(matrix, predicted, min_r, max_r):
    """RMSE on observed entries, rescaled to original purchase-count units."""
    mask = (matrix != -1).float()
    n    = mask.sum()
    if n == 0:
        return float("nan")
    scale = (max_r - min_r) if max_r > min_r else 1.0
    pred_scaled = predicted * scale + min_r
    true_scaled = matrix    * scale + min_r
    se = ((pred_scaled - true_scaled) ** 2) * mask
    return math.sqrt((se.sum() / n).item())


print("SVD, SVDLoss, compute_rmse defined.")

SVD, SVDLoss, compute_rmse defined.


## 4. Parameter Tuning -- Grid Search

Exhaustively trains every combination in `LR_GRID × LAM_GRID × LATENT_GRID` and selects the combo with the lowest best validation RMSE. Best values are written into `lr`, `lam`, `latent_vectors` so the training cell below picks them up automatically.

Run this cell **once** after the matrices are built (cells 3–5), then run the training cell.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Parameter Tuning -- Grid Search on train/val matrices
# ══════════════════════════════════════════════════════════════════════
import itertools

# ── Grid ─────────────────────────────────────────────────────────────
LR_GRID     = [0.001, 0.01,  0.05]
LAM_GRID    = [0.01,  0.1,   0.3 ]   # shared regularisation weight
LATENT_GRID = [10,    20,    40  ]   # latent dimension K
TUNE_EPOCHS = 50                     # max epochs per combo
TUNE_PAT    = 5                      # early-stop patience

param_grid = list(itertools.product(LR_GRID, LAM_GRID, LATENT_GRID))
print(f"Grid search: {len(param_grid)} combinations x up to {TUNE_EPOCHS} epochs")
print(f"{'#':>3}  {'lr':>7}  {'lam':>6}  {'K':>4}  {'best val RMSE':>14}")
print("-" * 42)

grid_results = []

for _k, (_lr, _lam, _K) in enumerate(param_grid, 1):
    torch.manual_seed(0)

    # Fresh parameters
    _uf = torch.randn(n_users, _K, requires_grad=True)
    _vf = torch.randn(n_items, _K, requires_grad=True)
    _ub = torch.randn(n_users, requires_grad=True)
    _vb = torch.randn(n_items, requires_grad=True)
    for _p in [_uf, _vf, _ub, _vb]:
        _p.data.mul_(0.01)

    _model     = SVD()
    _loss_fn   = SVDLoss(lam_u=_lam, lam_v=_lam, lam_1=_lam, lam_2=_lam)
    _optimizer = torch.optim.Adam([_uf, _vf, _ub, _vb], lr=_lr)

    _best_val, _wait = float('inf'), 0

    for _ep in range(TUNE_EPOCHS):
        _optimizer.zero_grad()
        _pred = _model(_uf, _vf, _ub, _vb)
        _loss = _loss_fn(train_matrix, _pred, _uf, _vf, _ub, _vb)
        _loss.backward()
        _optimizer.step()

        with torch.no_grad():
            _pred_eval = _model(_uf, _vf, _ub, _vb)
            _val_rmse  = compute_rmse(val_matrix, _pred_eval, min_rating, max_rating)

        if _val_rmse < _best_val:
            _best_val, _wait = _val_rmse, 0
        else:
            _wait += 1
            if _wait >= TUNE_PAT:
                break

    grid_results.append((_best_val, _lr, _lam, _K))
    _marker = ' <--' if _best_val == min(r[0] for r in grid_results) else ''
    print(f"{_k:>3}  {_lr:>7.4f}  {_lam:>6.3f}  {_K:>4d}  "
          f"{_best_val:>14.4f}{_marker}")

# ── Best combo ───────────────────────────────────────────────────────
best_val_t, best_lr, best_lam, best_K = min(grid_results, key=lambda x: x[0])

print(f"\nBest val RMSE : {best_val_t:.4f}")
print(f"  lr            = {best_lr}")
print(f"  lam           = {best_lam}")
print(f"  latent_vectors = {best_K}")

# ── Write into training hyperparameters ──────────────────────────────
lr             = best_lr
lam            = best_lam
latent_vectors = best_K
print("\nHyperparameters updated. Run the training cell below.")


Grid search: 27 combinations x up to 50 epochs
  #       lr     lam     K   best val RMSE
------------------------------------------


## 5. Training


In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
latent_vectors = 20
num_epoch      = 100
lr             = 0.01
lam            = 0.1
patience       = 10     # early stopping patience on val RMSE

# ── Initialise parameters ─────────────────────────────────────────────────────
torch.manual_seed(0)
user_features  = torch.randn(n_users, latent_vectors, requires_grad=True)
item_features  = torch.randn(n_items, latent_vectors, requires_grad=True)
user_bias      = torch.randn(n_users, requires_grad=True)
item_bias      = torch.randn(n_items, requires_grad=True)
for p in [user_features, item_features, user_bias, item_bias]:
    p.data.mul_(0.01)

SVD_model = SVD()
svdloss   = SVDLoss(lam_u=lam, lam_v=lam, lam_1=lam, lam_2=lam)
optimizer = torch.optim.Adam(
    [user_features, item_features, user_bias, item_bias], lr=lr
)

# ── Training loop with early stopping on val RMSE ────────────────────────────
best_val_rmse  = float("inf")
best_state     = None
patience_count = 0

for epoch in range(num_epoch):
    optimizer.zero_grad()
    pred = SVD_model(user_features, item_features, user_bias, item_bias)
    loss = svdloss(train_matrix, pred,
                   user_features, item_features, user_bias, item_bias)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        pred_eval  = SVD_model(user_features, item_features, user_bias, item_bias)
        train_rmse = compute_rmse(train_matrix, pred_eval, min_rating, max_rating)
        val_rmse   = compute_rmse(val_matrix,   pred_eval, min_rating, max_rating)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | loss: {loss.item():.3f} "
              f"| train RMSE: {train_rmse:.4f} | val RMSE: {val_rmse:.4f}")

    # Early stopping
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_state    = (
            user_features.data.clone(),
            item_features.data.clone(),
            user_bias.data.clone(),
            item_bias.data.clone(),
        )
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

# Restore best parameters
with torch.no_grad():
    user_features.data.copy_(best_state[0])
    item_features.data.copy_(best_state[1])
    user_bias.data.copy_(best_state[2])
    item_bias.data.copy_(best_state[3])

print(f"\nBest val RMSE: {best_val_rmse:.4f}")

## 6. Test Evaluation


In [ ]:
with torch.no_grad():
    pred_test = SVD_model(user_features, item_features, user_bias, item_bias)
    test_rmse = compute_rmse(test_matrix, pred_test, min_rating, max_rating)

print(f"Test RMSE : {test_rmse:.4f}")
print(f"Best val RMSE : {best_val_rmse:.4f}")

## 7. MAE (additional metric)


In [ ]:
with torch.no_grad():
    pred_test  = SVD_model(user_features, item_features, user_bias, item_bias)
    mask       = (test_matrix != -1).float()
    n_obs      = mask.sum()
    pred_scale = pred_test  * (max_rating - min_rating) + min_rating
    true_scale = test_matrix * (max_rating - min_rating) + min_rating
    test_mae   = (torch.abs(pred_scale - true_scale) * mask).sum() / n_obs

print(f"Test MAE  : {test_mae.item():.4f}")
print(f"Test RMSE : {test_rmse:.4f}")